In [1]:



import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

#  LOAD DATA (OPTIMIZED)

df = pd.read_csv("../data/vendor_data_big.csv", low_memory=False)

print("✅ Data Loaded")
print("Shape:", df.shape)

# CLEAN COLUMN NAMES

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("\nColumns standardized")


# REMOVE DUPLICATES

initial_rows = df.shape[0]
df = df.drop_duplicates()
print(f"\nRemoved {initial_rows - df.shape[0]} duplicate rows")

# 5. DATA TYPE OPTIMIZATION

for col in df.select_dtypes(include='object').columns:
    if df[col].nunique() < 50:
        df[col] = df[col].astype('category')

print("\nOptimized data types")


# 6. MISSING VALUE HANDLING


# Numeric → median
num_cols = df.select_dtypes(include=np.number).columns
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Categorical → mode
cat_cols = df.select_dtypes(include='category').columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print("\nMissing values handled")


#  OUTLIER HANDLING (FAST CLIPPING)

for col in num_cols:
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)
    df[col] = df[col].clip(lower, upper)

print("\nOutliers capped (1%-99%)")


# AUTO FEATURE DETECTION

def find_col(keywords):
    for col in df.columns:
        for k in keywords:
            if k in col:
                return col
    return None

revenue_col = find_col(['revenue','sales','amount'])
cost_col = find_col(['cost','purchase','price'])
inventory_col = find_col(['inventory','stock'])
vendor_col = find_col(['vendor','supplier'])

print("\nDetected Columns:")
print("Revenue:", revenue_col)
print("Cost:", cost_col)
print("Inventory:", inventory_col)
print("Vendor:", vendor_col)


# FEATURE ENGINEERING


# Profit & margin
if revenue_col and cost_col:
    df['profit'] = df[revenue_col] - df[cost_col]
    df['profit_margin'] = df['profit'] / (df[revenue_col] + 1)

# Inventory turnover
if revenue_col and inventory_col:
    df['inventory_turnover'] = df[revenue_col] / (df[inventory_col] + 1)

# Cost efficiency
if revenue_col and cost_col:
    df['cost_ratio'] = df[cost_col] / (df[revenue_col] + 1)

# Log transform (for skewed features)
for col in num_cols:
    if df[col].skew() > 1:
        df[f"log_{col}"] = np.log1p(df[col])

print("\nFeature engineering completed")


# BASIC VALIDATION CHECKS

print("\nValidation Checks:")

if 'profit' in df.columns:
    print("Profit min/max:", df['profit'].min(), df['profit'].max())

print("Any nulls left:", df.isnull().sum().sum())

#  FINAL PREVIEW

print("\nFinal Shape:", df.shape)
print("\nSample Data:")
print(df.head())


# SAVE CLEAN DATA

df.to_csv("../data/cleaned_data.csv", index=False)

print("\n✅ Cleaned data saved successfully")

✅ Data Loaded
Shape: (10000, 32)

Columns standardized

Removed 9900 duplicate rows

Optimized data types

Missing values handled

Outliers capped (1%-99%)

Detected Columns:
Revenue: None
Cost: None
Inventory: None
Vendor: vendorid

Feature engineering completed

Validation Checks:
Any nulls left: 0

Final Shape: (100, 39)

Sample Data:
  vendorid            vendorname       category         region  country  \
0     V001       AlphaSupply Co.  Raw Materials  North America      USA   
1     V002     BetaLogistics Ltd      Logistics         Europe  Germany   
2     V003  Gamma Tech Solutions    IT Services   Asia Pacific    India   
3     V004   Delta Packaging Inc      Packaging  North America   Canada   
4     V005     Epsilon Chemicals  Raw Materials    Middle East      UAE   

  contractstartdate contractenddate  contractvalue_usd  actualspend_usd  \
0        2022-01-01      2024-12-31           500000.0         487500.0   
1        2021-06-01      2024-05-31           323200.0     